In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
#!/usr/bin/env python3
"""
Knee MRI corpus-builder experiment harness -- raw DICOM -> fixed-size corpus.

Reproduces the preprocessing APPROACH from the reference notebook you
uploaded (knee_mri_twelve_findings_from_a_single_model.py, 0.924 public LB,
single model, no ensembling): the same slot/plane/fluid-sensitivity series
selection order, the same wide 6-94% within-series span (its docstring notes
this matters because the collateral ligaments and lateral meniscus sit in
the peripheral slices a tighter 15-85% span throws away), the same per-
series 2nd/98th-percentile intensity normalization, and the same physical-
millimetre crop computed from each series' own PixelSpacing before resizing.

The difference: that notebook is a single-study, fixed-constant INFERENCE
decoder. This is a BATCH, CONFIGURABLE corpus builder -- every number in
CorpusConfig below (slice counts per slot, crop size, stored resolution,
span, percentiles) is a knob, so you can run controlled corpus experiments
(e.g. "does 130mm beat 140mm crop", "does 336px beat 224px stored
resolution") instead of committing to one fixed layout per rebuild.

DEFAULTS match the reference notebook exactly: 64 slices across 5 slots
(18/14/12/8/12 = Sagittal-FS/Sagittal-nonFS/Coronal-FS/Coronal-nonFS/Axial),
CROP_MM=140.0, span 6-94%, percentiles 2-98. The one deliberate default
change is the stored resolution (img=224 instead of the reference's 336) --
see the note in CorpusConfig.img below, since it's a real trade-off, not a
free choice.

OUTPUT FORMAT matches the existing training corpus exactly:
    <out_dir>/all_vols.npy   (N, MAXS, IMG, IMG) uint8
    <out_dir>/all_masks.npy  (N, MAXS) uint8            -- 1 = real data, 0 = zero-padded
    <out_dir>/all_ids.npy    (N,) StudyInstanceUID strings, same row order
    <out_dir>/manifest.json  the exact config used + coverage stats, for the record
memory-mapped the same way knee_train_multiplane.py already reads Raptor's
corpus. This builder writes a single part (no extra_*.npy) -- see the loader
note at the very bottom for wiring it into the existing training config.

Does NOT touch labels. Same division of responsibility as the existing
corpus: this only produces pixel/mask/id arrays; labels are joined by
StudyInstanceUID downstream (train.csv / your LLM soft-label CSV), same as
before.

ASSUMPTION to verify on your end: train_series.csv is assumed to carry the
same Anatomical_Plane / Fluid_Sensitive / SeriesInstanceUID columns as
test_series.csv does in your existing inference scripts. Check this against
the real file before a full run -- a column-name mismatch will surface
immediately as every study falling back to empty slots, which the printed
per-slot coverage stats (and empty_studies count) will make obvious.

Running an experiment
----------------------
From a notebook cell (Kaggle/Colab) -- this is what you want, since argparse
and a notebook kernel's own launch arguments don't mix:

    from knee_corpus_builder import build_corpus

    build_corpus(limit=50, tag="pilot")                        # quick pilot, defaults
    build_corpus(crop_mm=130, limit=50, tag="crop130")          # audit-finding crop
    build_corpus(img=336, limit=50, tag="img336")                # reference notebook's own res
    build_corpus(tag="full", workers=4)                          # full build once you're happy

From a terminal instead:
    python knee_corpus_builder.py --limit 50 --tag pilot
    python knee_corpus_builder.py --crop_mm 130 --tag crop130 --limit 50
    python knee_corpus_builder.py --img 336 --tag img336 --limit 50
    python knee_corpus_builder.py --crop_mm 130 --img 224 --tag full --workers 4
"""

import os, sys, glob, json, time, argparse, multiprocessing as mp
from dataclasses import dataclass, field, asdict
import numpy as np
import pandas as pd


# ============================================================================
# Experiment config -- every number you asked to be able to adjust
# ============================================================================
@dataclass
class CorpusConfig:
    # (plane, fluid_sensitive(-1=don't care/0=no/1=yes), slice_count). Order
    # fixes where each slot lands along the MAXS axis -- same convention as
    # the reference notebook's SLOTS.
    # slots: list = field(default_factory=lambda: [
    #     ("Sagittal", 1, 18), ("Sagittal", 0, 14), ("Coronal", 1, 12),
    #     ("Coronal", 0, 8), ("Axial", -1, 12),
    # ])
    slots: list = field(default_factory=lambda: [
        ("Sagittal", 1, 22), ("Sagittal", 0, 18), ("Coronal", 1, 15),
        ("Coronal", 0, 10), ("Axial", -1, 15),
    ])
    crop_mm: float = 135.0   # physical crop box, mm. Reference notebook's default.
                              # An independent audit of this competition's real
                              # PixelSpacing/FOV distribution found 130mm covers
                              # 99.6% of series vs 140mm's 94.9% -- worth an
                              # experiment run, just not the default here since
                              # you asked to match the reference notebook first.
    img: int = 288            # stored per-slice resolution (H=W=img). The
                              # reference notebook caches at 336 and resizes
                              # UP to 384 at model-input time; storing at 224
                              # is smaller/faster to build and matches your
                              # existing model's res=224, but caps how much
                              # detail is available if you later want to try
                              # a higher-resolution model -- upsampling from a
                              # 224px cache can't recover it. Bump to 336 if
                              # you want that headroom; it's one flag.
    span_lo: float = 0.03     # within-series slice-pick span. Matches the
    span_hi: float = 0.97     # reference notebook's stated rationale (see docstring).
    pct_lo: float = 2.0       # per-series intensity-normalization percentiles
    pct_hi: float = 98.0
    limit: int = 0             # 0 = all studies; >0 = first N, for a quick experiment
    workers: int = 4
    out_dir: str = "/kaggle/working/corpus_experiment"
    tag: str = "exp1"

    @property
    def maxs(self):
        return sum(k for _, _, k in self.slots)


# ============================================================================
# DICOM reading -- verbatim approach from the reference notebook / your own
# build_study(), parametrized instead of module-level constants.
# ============================================================================
def _make_reader():
    import pydicom, cv2
    from pydicom.pixel_data_handlers.util import apply_modality_lut

    def order_and_meta(sdir):
        fs = glob.glob(sdir + "/*.dcm"); recs = []; ps_list = []
        for f in fs:
            try:
                h = pydicom.dcmread(f, stop_before_pixels=True)
                iop = getattr(h, 'ImageOrientationPatient', None)
                ipp = getattr(h, 'ImagePositionPatient', None)
                if iop is not None and ipp is not None and len(iop) == 6:
                    r = np.array(iop[:3], float); c = np.array(iop[3:], float)
                    n = np.cross(r, c); pos = float(np.dot(np.array(ipp, float), n))
                else:
                    pos = float(getattr(h, 'InstanceNumber', 0) or 0)
                ps = getattr(h, 'PixelSpacing', None); ps = float(ps[0]) if ps is not None else 0.5
                ps_list.append(ps); recs.append((pos, f, ps))
            except Exception:
                recs.append((0.0, f, 0.5))
        recs.sort(key=lambda x: x[0])
        med_ps = float(np.median(ps_list)) if ps_list else 0.5
        return [(f, ps) for _, f, ps in recs], med_ps

    def read_px(f):
        d = pydicom.dcmread(f)
        a = apply_modality_lut(d.pixel_array, d).astype(np.float32)
        if str(getattr(d, 'PhotometricInterpretation', '')) == 'MONOCHROME1':
            a = a.max() - a
        return a

    def mm_crop_resize(a, ps, crop_mm, img):
        h, w = a.shape; cpx = int(round(crop_mm / max(ps, 1e-3)))
        cpx = min(cpx, min(h, w)); y0 = (h - cpx) // 2; x0 = (w - cpx) // 2
        a = a[y0:y0 + cpx, x0:x0 + cpx]
        return cv2.resize(a, (img, img), interpolation=cv2.INTER_AREA)

    return order_and_meta, read_px, mm_crop_resize


def _pick_series_for_slot(rows, plane, fluid, used):
    cands = [r for r in rows if r['Anatomical_Plane'] == plane and r['SeriesInstanceUID'] not in used]
    if fluid in (0, 1):
        pref = [r for r in cands if int(r.get('Fluid_Sensitive', 0) or 0) == fluid]
        if pref:
            return pref[0]
    return cands[0] if cands else None


def build_study(sid, ser_records, dicom_dir, reader, cfg: CorpusConfig):
    """Same logic as the reference notebook's build_study(): fill cfg.maxs
    slices across cfg.slots, in order, leaving a slot as zeros (and its span
    of the mask as 0) if the study has no matching series left."""
    order_and_meta, read_px, mm_crop_resize = reader
    rows = ser_records.get(sid, [])
    vol = np.zeros((cfg.maxs, cfg.img, cfg.img), np.uint8); idx = 0; used = set()
    for plane, fluid, k in cfg.slots:
        r = _pick_series_for_slot(rows, plane, fluid, used)
        if r is None:
            idx += k; continue
        used.add(r['SeriesInstanceUID'])
        files, med_ps = order_and_meta(f"{dicom_dir}/{sid}/{r['SeriesInstanceUID']}")
        if not files:
            idx += k; continue
        n = len(files)
        lo, hi = int(n * cfg.span_lo), int(n * cfg.span_hi) - 1; hi = max(hi, lo)
        picks = np.linspace(lo, hi, k).round().astype(int) if n > 1 else [0] * k
        arrs = []; pss = []
        for p in picks:
            fp, ps = files[min(p, n - 1)]
            try:
                arrs.append(read_px(fp)); pss.append(ps)
            except Exception:
                arrs.append(None); pss.append(med_ps)
        valid = [a for a in arrs if a is not None]
        if valid:
            allpx = np.concatenate([a.ravel() for a in valid])
            loq, hiq = np.percentile(allpx, [cfg.pct_lo, cfg.pct_hi])
        else:
            loq, hiq = 0.0, 1.0
        for a, ps in zip(arrs, pss):
            if idx >= cfg.maxs: break
            if a is None: idx += 1; continue
            aw = np.clip((a - loq) / (hiq - loq + 1e-6), 0, 1)
            aw = mm_crop_resize(aw, ps if ps > 0 else med_ps, cfg.crop_mm, cfg.img)
            vol[idx] = (aw * 255).astype(np.uint8); idx += 1
        if idx >= cfg.maxs: break
    mask = (vol.reshape(cfg.maxs, -1).sum(1) > 0).astype(np.uint8)
    return vol, mask


# ============================================================================
# Batch driver -- multiprocessing over studies, writing straight into an
# on-disk memmap so peak RAM never holds the whole corpus at once.
# ============================================================================
_WORKER = {}


def _worker_init(cfg_dict, ser_records, dicom_dir):
    _WORKER["cfg"] = CorpusConfig(**cfg_dict)
    _WORKER["reader"] = _make_reader()
    _WORKER["ser"] = ser_records
    _WORKER["dicom_dir"] = dicom_dir


def _build_one(sid):
    cfg, reader = _WORKER["cfg"], _WORKER["reader"]
    ser, dicom_dir = _WORKER["ser"], _WORKER["dicom_dir"]
    try:
        vol, mask = build_study(sid, ser, dicom_dir, reader, cfg)
    except Exception as e:
        vol = np.zeros((cfg.maxs, cfg.img, cfg.img), np.uint8)
        mask = np.zeros((cfg.maxs,), np.uint8)
        print(f"  {sid[:16]} FAILED ({type(e).__name__}: {e}) -- wrote empty", flush=True)
    return sid, vol, mask


def find_train_root():
    cands = ["/kaggle/input/competitions/rsna-knee-abnormality-detection",
             "/kaggle/input/rsna-knee-abnormality-detection"]
    for b in cands:
        if os.path.exists(b + "/train.csv"):
            return b
    for d, _, f in os.walk("/kaggle/input"):
        if "train.csv" in f and (os.path.isdir(d + "/train_series") or os.path.isdir(d + "/train_images")):
            return d
    for d, _, f in os.walk("/kaggle/input"):
        if "train.csv" in f:
            return d
    raise RuntimeError("no train root under /kaggle/input")


def run(cfg: CorpusConfig):
    t0 = time.time()
    os.makedirs(cfg.out_dir, exist_ok=True)

    ROOT = find_train_root()
    tdir = ROOT + "/train_series"
    if not os.path.isdir(tdir):
        tdir = ROOT + "/train_images"
    print(f"train root: {ROOT} | series dir: {tdir}", flush=True)

    train = pd.read_csv(ROOT + "/train.csv").head(2407)
    train["StudyInstanceUID"] = train["StudyInstanceUID"].astype(str)
    ids = train["StudyInstanceUID"].tolist()
    if cfg.limit:
        ids = ids[:cfg.limit]
    N = len(ids)

    tser = pd.read_csv(ROOT + "/train_series.csv")
    tser["StudyInstanceUID"] = tser["StudyInstanceUID"].astype(str)
    tser["SeriesInstanceUID"] = tser["SeriesInstanceUID"].astype(str)
    SER = {k: v.to_dict("records") for k, v in tser.groupby("StudyInstanceUID")}

    print(f"building corpus [{cfg.tag}] | studies {N} | slots {cfg.slots} | maxs {cfg.maxs} | "
          f"crop_mm {cfg.crop_mm} | img {cfg.img} | span [{cfg.span_lo},{cfg.span_hi}] | "
          f"pct [{cfg.pct_lo},{cfg.pct_hi}] | workers {cfg.workers}", flush=True)
    est_gb = N * cfg.maxs * cfg.img * cfg.img / 1e9
    print(f"estimated all_vols.npy size: {est_gb:.2f} GB", flush=True)

    vols_path = os.path.join(cfg.out_dir, "all_vols.npy")
    masks_path = os.path.join(cfg.out_dir, "all_masks.npy")
    ids_path = os.path.join(cfg.out_dir, "all_ids.npy")

    vols = np.lib.format.open_memmap(vols_path, mode="w+", dtype=np.uint8,
                                      shape=(N, cfg.maxs, cfg.img, cfg.img))
    masks = np.lib.format.open_memmap(masks_path, mode="w+", dtype=np.uint8,
                                       shape=(N, cfg.maxs))

    cfg_dict = asdict(cfg)
    id_to_row = {sid: i for i, sid in enumerate(ids)}

    with mp.Pool(processes=cfg.workers, initializer=_worker_init,
                 initargs=(cfg_dict, SER, tdir)) as pool:
        done = 0
        for sid, vol, mask in pool.imap_unordered(_build_one, ids, chunksize=4):
            row = id_to_row[sid]
            vols[row] = vol
            masks[row] = mask
            done += 1
            if done % 100 == 0 or done == N:
                print(f"  {done}/{N} | {time.time()-t0:.0f}s", flush=True)

    vols.flush(); masks.flush()
    np.save(ids_path, np.array(ids, dtype=object))

    empty = int((masks.sum(axis=1) == 0).sum())
    per_slot_coverage = {}
    lo = 0
    for plane, fluid, k in cfg.slots:
        hi = lo + k
        cov = float((masks[:, lo:hi].sum(axis=1) > 0).mean())
        per_slot_coverage[f"{plane}|fluid={fluid}"] = round(cov, 4)
        lo = hi

    manifest = {**cfg_dict, "n_studies": N, "empty_studies": empty,
                "per_slot_study_coverage": per_slot_coverage,
                "elapsed_s": round(time.time() - t0, 1)}
    with open(os.path.join(cfg.out_dir, f"manifest_{cfg.tag}.json"), "w") as f:
        json.dump(manifest, f, indent=2)

    print(f"\nDONE {time.time()-t0:.0f}s", flush=True)
    print(f"wrote {vols_path} ({vols.nbytes/1e9:.2f} GB), {masks_path}, {ids_path}", flush=True)
    print(f"empty studies: {empty}/{N}", flush=True)
    print(f"per-slot study coverage: {per_slot_coverage}", flush=True)
    if empty > N * 0.05:
        print("[warn] >5% of studies have an entirely empty corpus row -- check the "
              "train_series.csv column-name assumption noted in the module docstring "
              "before trusting this build.", flush=True)


def parse_args():
    p = argparse.ArgumentParser(description="Build a knee-MRI training corpus (experiment harness).")
    p.add_argument("--crop_mm", type=float, default=None)
    p.add_argument("--img", type=int, default=None)
    p.add_argument("--span_lo", type=float, default=None)
    p.add_argument("--span_hi", type=float, default=None)
    p.add_argument("--pct_lo", type=float, default=None)
    p.add_argument("--pct_hi", type=float, default=None)
    p.add_argument("--limit", type=int, default=None)
    p.add_argument("--workers", type=int, default=None)
    p.add_argument("--out_dir", type=str, default=None)
    p.add_argument("--tag", type=str, default=None)
    p.add_argument("--slots", type=str, default=None,
                    help='JSON list of [plane, fluid, k], e.g. '
                         '\'[["Sagittal",1,18],["Sagittal",0,14],["Coronal",1,12],'
                         '["Coronal",0,8],["Axial",-1,12]]\'')
    # parse_known_args, not parse_args: a Jupyter/Colab/Kaggle kernel launches
    # this process with its OWN "-f <connection_file>" argument, which a
    # strict parse_args() rejects with SystemExit. Silently ignoring unknown
    # args is exactly what a notebook context needs here; from a real
    # terminal, a genuine typo'd flag still just gets silently dropped rather
    # than erroring, so double-check `args` if a run doesn't do what you expect.
    args, _unknown = p.parse_known_args()
    return args


def build_corpus(**overrides):
    """Notebook-friendly entry point -- bypasses argparse entirely, so it
    never sees the kernel's own -f <connection_file> argument in the first
    place. This is the recommended way to run from a Kaggle/Colab cell; pass
    any CorpusConfig field as a keyword, e.g.:

        build_corpus(crop_mm=130, img=224, limit=50, tag="pilot")

        # slots also accepts the JSON-string form the CLI uses, or a plain
        # list of tuples/lists directly:
        build_corpus(slots=[("Sagittal",1,18),("Sagittal",0,14),
                             ("Coronal",1,12),("Coronal",0,8),("Axial",-1,12)],
                     tag="full")

    Returns whatever run() returns (nothing currently -- results land in
    cfg.out_dir, same as the CLI path).
    """
    cfg = CorpusConfig()
    for name, val in overrides.items():
        if not hasattr(cfg, name):
            raise TypeError(f"unknown CorpusConfig field: {name!r} -- valid fields: "
                             f"{list(asdict(cfg).keys())}")
        if name == "slots" and isinstance(val, str):
            val = [tuple(x) for x in json.loads(val)]
        elif name == "slots":
            val = [tuple(x) for x in val]
        setattr(cfg, name, val)
    return run(cfg)


def main():
    args = parse_args()
    cfg = CorpusConfig()
    for name in ("crop_mm", "img", "span_lo", "span_hi", "pct_lo", "pct_hi",
                 "limit", "workers", "out_dir", "tag"):
        val = getattr(args, name)
        if val is not None:
            setattr(cfg, name, val)
    if args.slots is not None:
        cfg.slots = [tuple(x) for x in json.loads(args.slots)]
    run(cfg)


if __name__ == "__main__":
    main()


# ============================================================================
# Using this corpus from knee_train_multiplane.py
#
# _open_corpus() there merges TWO parts via _TwoPartVols, because Raptor's
# corpus happened to ship split across two Kaggle dataset uploads. This
# builder produces ONE part, so swap in a single-part loader rather than
# feeding it through the two-part merge:
#
#   def _open_corpus():
#       vols = np.load("<out_dir>/all_vols.npy", mmap_mode="r")
#       masks = np.load("<out_dir>/all_masks.npy")
#       return vols, masks
#
#   def _open_ids():
#       return np.load("<out_dir>/all_ids.npy", allow_pickle=True).astype(str)
#
# Everything downstream (id2row, StudyWindowsMultiPlane, SLOT_DEF-based
# windowing) is unchanged -- it only ever calls _open_corpus()/_open_ids(),
# never the two-part machinery directly. One thing worth checking when you
# do this: this builder's cfg.slots order/widths must match knee_train_
# multiplane.py's SLOT_DEF (currently 12/10/8/6/8 across the same 5 planes)
# for the windowing to stay boundary-safe -- update SLOT_DEF there if you
# change --slots here.
# ============================================================================

train root: /kaggle/input/competitions/rsna-knee-abnormality-detection | series dir: /kaggle/input/competitions/rsna-knee-abnormality-detection/train_series
building corpus [exp1] | studies 2407 | slots [('Sagittal', 1, 22), ('Sagittal', 0, 18), ('Coronal', 1, 15), ('Coronal', 0, 10), ('Axial', -1, 15)] | maxs 80 | crop_mm 135.0 | img 288 | span [0.03,0.97] | pct [2.0,98.0] | workers 4
estimated all_vols.npy size: 15.97 GB
  100/2407 | 77s
  200/2407 | 150s
  300/2407 | 221s
  400/2407 | 298s
  500/2407 | 365s
  600/2407 | 442s
  700/2407 | 517s
  800/2407 | 592s
  900/2407 | 654s
  1000/2407 | 719s
  1100/2407 | 785s
  1200/2407 | 849s
  1300/2407 | 955s
  1400/2407 | 1021s
  1500/2407 | 1119s
  1600/2407 | 1186s
  1700/2407 | 1257s
  1800/2407 | 1321s
  1900/2407 | 1397s
  2000/2407 | 1463s
  2100/2407 | 1534s
  2200/2407 | 1605s
  2300/2407 | 1685s
  2400/2407 | 1766s
  2407/2407 | 1771s

DONE 1772s
wrote /kaggle/working/corpus_experiment/all_vols.npy (15.97 GB), /kaggle/working/cor